In [1]:
device = "cuda:10"

In [2]:
import math
import random
import itertools

import torch
from torch import Tensor
import tensordict
import torch.nn.functional as F
import transformers
import plotly.express
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import IPython.display
import sudoku

In [25]:
puzzle = sudoku.Sudoku(3, difficulty=0.5)
solution = puzzle.solve()

def tokenize(puzzles: list[sudoku.Sudoku]) -> tensordict.TensorDict:
    boards = [[[cell or 0 for cell in row] for row in puzzle.board] for puzzle in puzzles]
    return tensordict.TensorDict(
        x=torch.tensor(boards, dtype=torch.long),
        y=torch.stack([torch.tensor(puzzle.solve().board, dtype=torch.long) for puzzle in puzzles])
    )

tokenized = tokenize([
    sudoku.Sudoku(3, difficulty=0.1, seed=random.Random().randint(0, 10000000)) for _ in range(10)
])

In [29]:
tokenized["x"][0]

tensor([[0, 0, 0, 0, 0, 6, 0, 0, 0],
        [0, 2, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 9],
        [0, 0, 0, 0, 0, 0, 7, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 8, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 5, 0, 0, 0, 0],
        [0, 0, 0, 4, 0, 0, 0, 0, 0],
        [0, 0, 3, 0, 0, 0, 0, 0, 0]])

In [32]:
def precompute_rope(seq_len: int, head_dim: int, base: float = 10_000.0):
    half  = head_dim // 2
    theta = 1.0 / (base ** (torch.arange(0, half).float() / half))    # (half,)
    pos   = torch.arange(seq_len).float()                             # (L,)
    freqs = torch.outer(pos, theta)                                   # (L, half)
    freqs = torch.cat([freqs, freqs], dim=-1)                         # (L, head_dim)
    return freqs.cos(), freqs.sin()

def rotate_half(x: Tensor) -> Tensor:
    half = x.shape[-1] // 2
    return torch.cat([-x[..., half:], x[..., :half]], dim=-1)

def apply_rope(x: Tensor, cos: Tensor, sin: Tensor) -> Tensor:
    # x: (B, n_heads, L, head_dim), cos/sin: (L, head_dim)
    cos = cos[None, None]  # (1, 1, L, head_dim)
    sin = sin[None, None]
    return x * cos + rotate_half(x) * sin

class SelfAttention(torch.nn.Module):
    def __init__(self, d_model: int, n_heads: int, is_causal: bool, max_seq_len: int, rope_base: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.c_attn = torch.nn.Linear(d_model, 3 * d_model, bias=False)
        self.c_proj = torch.nn.Linear(d_model, d_model, bias=False)
        self.n_heads = n_heads
        self.d_model = d_model
        self.is_causal = is_causal
        rope_cos, rope_sin = precompute_rope(max_seq_len, d_model // n_heads, base=rope_base)
        self.rope_cos = torch.nn.Buffer(rope_cos)
        self.rope_sin = torch.nn.Buffer(rope_sin)

    def forward(self, x: Tensor) -> Tensor:
        B, T, C = x.size()
        qkv: Tensor = self.c_attn(x)
        q, k, v = qkv.split(self.d_model, dim=2)
        q = q.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        k = k.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        v = v.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
        k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])
        y = F.scaled_dot_product_attention(q, k, v, is_causal=self.is_causal)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class FeedForward(torch.nn.Module):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        self.fc1 = torch.nn.Linear(d_model, hidden_dim, bias=False)
        self.fc2 = torch.nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class TransformerBlock(torch.nn.Module):
    def __init__(self, d_model: int, n_heads: int, ff_ratio: int, is_causal: bool, max_seq_len: int, rope_base: float, dropout: float):
        super().__init__()
        self.norm1 = torch.nn.RMSNorm(d_model, eps=1e-6)
        self.dropout1 = torch.nn.Dropout(dropout)
        self.attn = SelfAttention(d_model, n_heads, is_causal, max_seq_len, rope_base)
        self.norm2 = torch.nn.RMSNorm(d_model, eps=1e-6)
        self.mlp = FeedForward(d_model, ff_ratio * d_model)
        self.dropout2 = torch.nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout1(self.attn(self.norm1(x)))
        x = x + self.dropout2(self.mlp(self.norm2(x)))
        return x

class Transformer(torch.nn.Module):
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        ff_ratio: int,
        n_layers: int,
        is_causal: bool,
        max_seq_len: int,
        rope_base: float,
        dropout: float,
    ):
        super().__init__()
        self.layers = torch.nn.Sequential()
        for _ in range(n_layers):
            block = TransformerBlock(d_model, n_heads, ff_ratio, is_causal, max_seq_len, rope_base, dropout)
            self.layers.append(block)

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)

class TimestepEmbedder(torch.nn.Module):
    """
    Embeds scalar timesteps into a vector space using sinusoidal positional encodings.
    """
    def __init__(self, hidden_size: int, frequency_embedding_size: int = 256):
        super().__init__()
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(frequency_embedding_size, hidden_size, bias=True),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_size, hidden_size, bias=True),
        )
        self.frequency_embedding_size = frequency_embedding_size

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # t is expected to be shape (B,)
        half = self.frequency_embedding_size // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(0, half, dtype=torch.float32, device=t.device) / half
        )
        # (B, half)
        args = t[:, None].float() * freqs[None]
        # (B, frequency_embedding_size)
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        
        return self.mlp(embedding) # (B, hidden_size)


class DiffusionTransformerLM(torch.nn.Module):
    class Output(tensordict.TensorClass):
        x_out: Tensor
        logits: Tensor

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        ff_ratio: int,
        n_layers: int,
        is_causal: bool,
        max_seq_len: int,
        rope_base: float,
        emb_dim: int,
        dropout: float,
        vocab_size: int,
        mask_token_id: int,
        bos_token_id: int,
        eos_token_id: int,
        pad_token_id: int,
    ):
        super().__init__()
        self.transformer = Transformer(d_model, n_heads, ff_ratio, n_layers, is_causal, max_seq_len, rope_base, dropout)
        self.token_emb = torch.nn.Embedding(vocab_size, emb_dim)
        self.time_emb = TimestepEmbedder(d_model)
        self.mask_token_id = mask_token_id
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id

    def forward(self, x: Tensor, t: Tensor, h: Tensor | None) -> Output:
        if not x.dtype.is_floating_point:
            x = self.token_emb(x)
        t_emb = self.time_emb(t)
        x = x + t_emb.unsqueeze(1)
        if h is not None:
            x = x + h
        x_out = self.transformer(x)
        logits = F.linear(x_out, self.token_emb.weight)
        return self.Output(x_out=x_out, logits=logits)

    def generate(self, inputs: Tensor | None, timesteps: Tensor, loophole: bool, n: int | None = None, max_len: int | None = None, temperature: float = 1.0) -> Tensor:
        """
        Generates a sequence by iteratively unmasking tokens using stochastic sampling.
        """
        if inputs is None and (n is None or max_len is None):
            raise ValueError("Must provide either inputs or n and max_len.")
        device = next(self.parameters()).device

        if inputs is None:
            assert n is not None and max_len is not None
            y = torch.full((n, max_len), self.mask_token_id, dtype=torch.long, device=device)
            y[:, 0] = self.bos_token_id
        else:
            y = inputs.to(device)
            max_len = y.shape[1]

        h: Tensor | None = None
        out: DiffusionTransformerLM.Output
        for i in range(len(timesteps) - 1):
            t = timesteps[i].unsqueeze(0).to(device)
            t_next = timesteps[i + 1].unsqueeze(0).to(device)

            with torch.no_grad():
                out = self(y, t, h)
                logits = out.logits
                h = out.x_out

            # --- 1. Stochastic Prediction (The Gumbel-Max Trick) ---
            # Generate standard Gumbel noise
            uniform_noise = torch.rand_like(logits)
            gumbel_noise = -torch.log(-torch.log(uniform_noise + 1e-9))
            
            # Scale logits by temperature and add the noise
            noisy_logits = (logits / temperature) + gumbel_noise
            
            # Take the argmax of the noisy logits. 
            # This is mathematically equivalent to sampling from the Categorical distribution!
            preds = torch.argmax(noisy_logits, dim=-1)
            
            # --- 2. Confidence Calculation ---
            # We must use the model's *actual* probability of the token it just sampled 
            # as the confidence score. 
            probs = torch.softmax(logits, dim=-1)
            scores = torch.gather(probs, 2, preds.unsqueeze(-1)).squeeze(-1)

            # --- 3. The Masking Schedule ---
            is_masked = (y == self.mask_token_id)
            num_keep_masked = int(t_next.item() * max_len)

            # Protect already unmasked tokens from being evaluated for re-masking
            scores = scores.masked_fill(~is_masked, torch.inf)

            # Temporarily accept all sampled predictions
            y = torch.where(is_masked, preds, y)

            # Re-mask the lowest confidence tokens
            if num_keep_masked > 0:
                _, mask_indices = torch.topk(scores, k=num_keep_masked, dim=-1, largest=False)
                y.scatter_(1, mask_indices, self.mask_token_id)

            if loophole:
                h[y != self.mask_token_id] = 0.0
            else:
                h = None
        return y


def q_sample(x0: Tensor, t: Tensor, mask_token_id: Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Mask each token independently with probability t."""
    mask = torch.rand_like(x0, dtype=torch.float, device=x0.device) < t.unsqueeze(-1)
    x_t  = x0.masked_fill(mask, mask_token_id)
    return x_t, mask


In [24]:
MASK_TOKEN_ID = 0
BOS_TOKEN_ID = 1
EOS_TOKEN_ID = 2
PAD_TOKEN_ID = 3

In [25]:
VOCAB_SIZE  = 10
SEQ_LEN     = 256
BATCH_SIZE  = 32
N_SAMPLES   = BATCH_SIZE * 1000

torch.manual_seed(42)

def make_repeating(n, seq_len, rand_padding=False):
    """Alternating pair: [a, b, a, b, ...]"""
    a = torch.randint(4, VOCAB_SIZE, (n, 1))
    b = torch.randint(4, VOCAB_SIZE, (n, 1))
    ab = torch.cat([a, b], dim=1).repeat(1, seq_len // 2)
    ab[:, 0] = BOS_TOKEN_ID
    if rand_padding:
        padding_len = torch.randint(0, seq_len // 4, (n,))
        for i in range(n):
            ab[i, seq_len - padding_len[i] - 1] = EOS_TOKEN_ID
            ab[i, seq_len - padding_len[i]:] = PAD_TOKEN_ID
    else:
        ab[:, -1] = EOS_TOKEN_ID
    return ab

def make_arithmetic(n, seq_len):
    """Arithmetic sequence starting at a random offset."""
    start = torch.randint(4, VOCAB_SIZE - seq_len, (n, 1))
    offsets = torch.arange(seq_len).unsqueeze(0)
    return (start + offsets) % VOCAB_SIZE

def make_constant(n, seq_len):
    """All tokens the same value."""
    v = torch.randint(4, VOCAB_SIZE, (n, 1))
    return v.expand(n, seq_len).clone()


train_data = make_repeating(N_SAMPLES, SEQ_LEN)
train_dataset    = TensorDataset(train_data)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

valid_data = make_repeating(BATCH_SIZE * 100, SEQ_LEN)

print(f"Dataset: {train_data.shape}  |  {len(train_dataloader)} batches")
print("Repeating sample :", train_data[1].tolist())

Dataset: torch.Size([32000, 256])  |  1000 batches
Repeating sample : [1, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 4, 9, 2]


In [26]:
model = DiffusionTransformerLM(
    d_model=128,
    n_heads=4,
    ff_ratio=4,
    n_layers=4,
    is_causal=False,
    max_seq_len=SEQ_LEN,
    rope_base=10_000.0,
    emb_dim=128,
    dropout=0.1,
    vocab_size=VOCAB_SIZE,
    mask_token_id=MASK_TOKEN_ID,
    bos_token_id=BOS_TOKEN_ID,
    eos_token_id=EOS_TOKEN_ID,
    pad_token_id=PAD_TOKEN_ID,
)

In [28]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
model.to(device).train()
mask_id = torch.tensor(model.mask_token_id, device=device)

history = []

out: DiffusionTransformerLM.Output
train_stream = itertools.cycle(train_dataloader)
train_stream = itertools.islice(train_stream, 1000+1)
for step, (x_0,) in enumerate(train_stream, start=1):
    x_0 = x_0.to(device)
    t = torch.rand(x_0.shape[0], device=device) * 1.05
    t = torch.clamp(t, max=1.0)
    x_t, mask = q_sample(x_0, t, mask_id)  # (B, L), (B, L) bool
    x_t[:, 0] = model.bos_token_id
    x_t[:, 1] = x_0[:, 1]  # always reveal the second token to make the task easier for the model
    x_t[:, 2] = x_0[:, 2]  # always reveal the third token to make the task easier for the model

    out = model(x_t, t, h=None)                                 # logits: (B, L, V)
    loss = F.cross_entropy(out.logits[mask], x_0[mask])   # only masked positions

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    train_loss = loss
    
    if step % 100 == 0:
        model.eval()
        x_0 = valid_data.to(device)
        t = torch.rand(x_0.shape[0], device=device) * 1.05
        t = torch.clamp(t, max=1.0)
        x_t, mask = q_sample(x_0, t, mask_id)
        x_t[:, 0] = model.bos_token_id
        with torch.no_grad():
            out = model(x_t, t, h=None)
            val_loss = F.cross_entropy(out.logits[mask], x_0[mask])
        history.append({
            "step": step,
            "train_loss": train_loss.item(),
            "val_loss": val_loss.item(),
        })
        IPython.display.clear_output(wait=True)
        plotly.express.line(pd.DataFrame(history), x="step", y=["train_loss", "val_loss"]).show()
        model.train()


In [29]:
pd.DataFrame(history)

,step,train_loss,val_loss
0,100,1.259387,1.858573
1,200,0.794391,1.035314
2,300,0.693661,0.795464
3,400,0.190605,0.409606
4,500,0.057388,0.442190
5,600,0.086215,0.606753
6,700,0.288509,0.387414
7,800,0.034646,0.602013
8,900,0.226295,0.814334
9,1000,0.104069,0.565826


In [38]:
accs = []
for steps in [1, 2, 5, 10, 20, 50, 100]:
    x_0 = valid_data[:100].to(device)
    x_t = torch.full_like(x_0, model.mask_token_id)
    x_t[:, 0] = model.bos_token_id
    x_t[:, 1] = x_0[:, 1]  # always reveal
    x_t[:, 2] = x_0[:, 2]  # always reveal
    timesteps = torch.linspace(1.0, 0.0, steps=steps)
    for loophole in [True, False]:
        pred = model.generate(x_t, timesteps=timesteps, temperature=1.0, loophole=loophole)
        relevant_mask = (x_0 != model.pad_token_id) & (x_0 != model.bos_token_id) & (x_0 != model.eos_token_id)
        accuracy = (pred[relevant_mask] == x_0[relevant_mask]).float().mean().item()
        accs.append({"iters": steps, "acc": accuracy, "loophole": loophole})
    IPython.display.clear_output(wait=True)
    viz = pd.DataFrame(accs).melt(id_vars=["iters", "loophole"], value_vars=["acc"], var_name="metric", value_name="accuracy")
    plotly.express.line(viz, x="iters", y="accuracy", color="loophole", markers=True).show()


In [46]:
real_tokens = list(set(range(VOCAB_SIZE)) - {model.mask_token_id, model.bos_token_id, model.eos_token_id, model.pad_token_id})

for x1 in real_tokens:
    for x2 in real_tokens:
        inputs = torch.full((1, SEQ_LEN), model.mask_token_id, dtype=torch.long, device=device)
        inputs[:, 0] = model.bos_token_id
        inputs[:, 1] = x1
        inputs[:, 2] = x2

        pred = model.generate(inputs, timesteps=torch.linspace(1.0, 0.0, steps=2), loophole=False, temperature=1.0)
        target = torch.empty_like(inputs)
        target[:, 1::2] = x1
        target[:, ::2] = x2
        target[:, 0] = model.bos_token_id
        target[:, -1] = model.eos_token_id


        acc = (pred == target).float().mean().item()
        # print(f"Input: {x1}, {x2} |  Accuracy: {acc:.2f}")


In [42]:
model = DiffusionTransformerLM(
    d_model=128,
    n_heads=4,
    ff_ratio=4,
    n_layers=4,
    is_causal=False,
    max_seq_len=SEQ_LEN,
    rope_base=10_000.0,
    emb_dim=128,
    dropout=0.1,
    vocab_size=VOCAB_SIZE,
    mask_token_id=MASK_TOKEN_ID,
    bos_token_id=BOS_TOKEN_ID,
    eos_token_id=EOS_TOKEN_ID,
    pad_token_id=PAD_TOKEN_ID,
)

In [44]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
model.to(device).train()
mask_id = torch.tensor(model.mask_token_id, device=device)

history = []

out: DiffusionTransformerLM.Output
train_stream = itertools.cycle(train_dataloader)
train_stream = itertools.islice(train_stream, 1000+1)
p = 0.5
rng = random.Random(42)

for step, (x_0,) in enumerate(train_stream, start=1):
    x_0 = x_0.to(device)
    t = torch.rand(x_0.shape[0], device=device) * 1.05
    t = torch.clamp(t, max=1.0)
    x_t, mask = q_sample(x_0, t, mask_id)  # (B, L), (B, L) bool
    x_t[:, 0] = model.bos_token_id
    x_t[:, 1] = x_0[:, 1]  # always reveal the second token to make the task easier for the model
    x_t[:, 2] = x_0[:, 2]  # always reveal the third token to make the task easier for the model

    # loopholing
    if rng.random() < p:
        with torch.no_grad():
            h = model(x_t, t, h=None).x_out.detach()
    else:
        h = None

    out = model(x_t, t, h=h)                                 # logits: (B, L, V)
    loss = F.cross_entropy(out.logits[mask], x_0[mask])   # only masked positions

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    train_loss = loss
    
    if step % 100 == 0:
        model.eval()
        x_0 = valid_data.to(device)
        t = torch.rand(x_0.shape[0], device=device) * 1.05
        t = torch.clamp(t, max=1.0)
        x_t, mask = q_sample(x_0, t, mask_id)
        x_t[:, 0] = model.bos_token_id
        with torch.no_grad():
            out = model(x_t, t, h=None)
            val_loss = F.cross_entropy(out.logits[mask], x_0[mask])
        history.append({
            "step": step,
            "train_loss": train_loss.item(),
            "val_loss": val_loss.item(),
        })
        IPython.display.clear_output(wait=True)
        plotly.express.line(pd.DataFrame(history), x="step", y=["train_loss", "val_loss"]).show()
        model.train()


In [45]:
accs = []
for steps in [1, 2, 5, 10, 20, 50, 100]:
    x_0 = valid_data[:100].to(device)
    x_t = torch.full_like(x_0, model.mask_token_id)
    x_t[:, 0] = model.bos_token_id
    x_t[:, 1] = x_0[:, 1]  # always reveal
    x_t[:, 2] = x_0[:, 2]  # always reveal
    timesteps = torch.linspace(1.0, 0.0, steps=steps)
    for loophole in [True, False]:
        pred = model.generate(x_t, timesteps=timesteps, temperature=1.0, loophole=loophole)
        relevant_mask = (x_0 != model.pad_token_id) & (x_0 != model.bos_token_id) & (x_0 != model.eos_token_id)
        accuracy = (pred[relevant_mask] == x_0[relevant_mask]).float().mean().item()
        accs.append({"iters": steps, "acc": accuracy, "loophole": loophole})
    IPython.display.clear_output(wait=True)
    viz = pd.DataFrame(accs).melt(id_vars=["iters", "loophole"], value_vars=["acc"], var_name="metric", value_name="accuracy")
    plotly.express.line(viz, x="iters", y="accuracy", color="loophole", markers=True).show()
